# UQRoute-TC — corrected Qwen 1.5B development coverage

Run only after reviewing the locally generated `coverage_report.json`. This diagnostic selects one constructed case per benchmark/source-file stratum, with at most 85 requests. The five clean cases intentionally overlap the earlier feasibility selection; this is a separate planned run, never a retry or pooled result. It does not replace the earlier evidence.

Select a Colab L4 GPU. Run cells in order. Enter the report's manifest SHA256 and row count, then upload `corrected_qwen15b_coverage_transfer_v1.zip` when prompted. Do not upload reference sidecars or raw benchmark data. A new checkout and output namespace preserve the prior feasibility run. Do not change the hash or selection to bypass validation.

The pinned tokenizer must fit every prompt plus the frozen completion cap within context before any server/generation startup. If a prompt does not fit, all dispatch stops; no truncation or row deletion is applied. Review GPU quota before running: runtime cannot be inferred from the five clean calls, and Colab quota is not read automatically. A run may be incomplete; prior durable intents are never resubmitted. Keep all outcomes and elapsed usage, including errors.

The notebook validates setup, applies the Ninja PATH fix, starts the pinned server, sends approved development requests, replays evidence, and downloads a new evidence ZIP. It never scores references, changes official labels, accesses held-out records, trains models, commits or pushes. Main-experiment readiness stays false.

Full GPU execution remains locally unverified; CPU tests and syntax checks are separate evidence.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
import hashlib
import json
import os
import re
import shutil
import signal
import socket
import subprocess
import sys
import time
import uuid
import zipfile
from urllib.request import build_opener, ProxyHandler

# Preserve earlier notebook state if this kernel already has a project server.
PRIOR_PROCESS = globals().get("SERVER_PROCESS")
PRIOR_LOG = globals().get("SERVER_LOG")

ENV = Path("/tmp/uqroute-vllm030-notebook-v2")
PY = ENV / "bin/python"
REPO = Path("/content/uqroute-tc-coverage-v1")
PREPARED = Path("/content/uqroute_prepared/corrected_qwen15b_coverage_v1")
ARTIFACTS = REPO / "artifacts/local_development_audits"
WORK = ARTIFACTS / "corrected_qwen15b_coverage_run_v1"
META = WORK / "metadata"
RUN_DIR = WORK / "run"
CODE_REVISION = "6dfed208793f0061b79dba76359b91d6ea484fcf"
EXPECTED_MANIFEST = input("Paste the reviewed coverage prepared_manifest_sha256: ").strip()
if not re.fullmatch(r"[0-9a-f]{64}", EXPECTED_MANIFEST):
    raise RuntimeError("A reviewed lowercase SHA256 is required; no generation started.")
APPROVED_REQUEST_COUNT = int(input("Enter rows from the reviewed coverage_report.json (5..85): "))
if not 5 <= APPROVED_REQUEST_COUNT <= 85:
    raise RuntimeError("Invalid approved request count.")
ENDPOINT = "http://127.0.0.1:8000/v1"
OPENER = build_opener(ProxyHandler({}))


def sha(raw):
    return hashlib.sha256(raw).hexdigest()


def canonical(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False,
                      separators=(",", ":"), allow_nan=False).encode("utf-8")


def json_write_new(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("x", encoding="utf-8") as stream:
        json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)


def child_json(script):
    result = subprocess.run([str(PY), "-c", script], capture_output=True, text=True)
    if result.returncode:
        print("\n".join((result.stdout + "\n" + result.stderr).splitlines()[-50:]))
        raise RuntimeError("Isolated interpreter check failed; see output above.")
    # Probe scripts print a unique marker so library startup messages are harmless.
    marker = "UQROUTE_JSON="
    marked = [line[len(marker):] for line in result.stdout.splitlines() if line.startswith(marker)]
    if len(marked) != 1:
        raise RuntimeError("Expected one structured probe report.")
    return json.loads(marked[0]), result.stdout + "\n" + result.stderr


def run_workflow(arguments):
    result = subprocess.run(
        [str(PY), str(REPO / "tools/development_audits/corrected_generation_v1/workflow.py"),
         "--repo-root", str(REPO), *arguments],
        env=SERVER_ENV, capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode:
        print(result.stderr)
        raise RuntimeError("Workflow stopped. Preserve existing records; do not bypass hashes.")
    return result


print("Step 1: inspecting the GPU")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True)
print(gpu.stdout or gpu.stderr)
if gpu.returncode or "L4" not in gpu.stdout:
    raise RuntimeError("This notebook requires the selected L4 runtime. Switch to L4 before continuing.")


## 2. Reuse or create the isolated environment

The environment uses vLLM 0.30.0. Environment creation bypasses ensurepip; Ninja is installed if absent and the executable PATH is set explicitly. Existing failed environment directories are preserved.


In [ ]:
identity = {"purpose": "uqroute-isolated-vllm-setup-v2", "vllm": "0.30.0",
            "python_major_minor": list(sys.version_info[:2])}
marker = ENV / "uqroute_setup_identity.json"

if ENV.exists() and (not marker.is_file() or json.loads(marker.read_text()) != identity):
    # Preserve the unrelated/partial directory and choose a fresh one automatically.
    ENV = Path("/tmp") / ("uqroute-vllm030-complete-" + uuid.uuid4().hex[:8])
    PY = ENV / "bin/python"
    marker = ENV / "uqroute_setup_identity.json"

if not ENV.exists():
    made = subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(ENV)],
                          capture_output=True, text=True)
    if made.returncode:
        print(made.stdout + made.stderr)
        raise RuntimeError("Environment creation failed; no existing directory was removed.")
    json_write_new(marker, identity)
if not PY.is_file():
    raise RuntimeError("Recognized environment is incomplete; preserve it.")

setup_logs = ENV / "uqroute_setup_logs"
setup_logs.mkdir(exist_ok=True)
probe = subprocess.run([str(PY), "-m", "pip", "--version"], capture_output=True, text=True)
if probe.returncode:
    bootstrap = subprocess.run([sys.executable, "-m", "pip", "--python", str(PY), "install", "pip"],
                               capture_output=True, text=True)
    json_write_new(setup_logs / ("bootstrap_" + uuid.uuid4().hex + ".json"),
                   {"returncode": bootstrap.returncode, "stdout": bootstrap.stdout, "stderr": bootstrap.stderr})
    if bootstrap.returncode:
        print(bootstrap.stdout + bootstrap.stderr)
        raise RuntimeError("Isolated pip bootstrap failed.")

installed = subprocess.run([str(PY), "-c", "from importlib.metadata import version; print(version('vllm'))"],
                           capture_output=True, text=True)
if installed.returncode == 0 and installed.stdout.strip() != "0.30.0":
    raise RuntimeError("Unexpected vLLM version; environment left unchanged.")
if installed.returncode:
    install_log = setup_logs / ("install_" + uuid.uuid4().hex + ".log")
    print("Installing vLLM 0.30.0. This may take several minutes.", flush=True)
    print("Log:", install_log, flush=True)
    with install_log.open("x", encoding="utf-8") as stream:
        installed = subprocess.run([str(PY), "-m", "pip", "install", "vllm==0.30.0"],
                                   stdout=stream, stderr=subprocess.STDOUT)
    if installed.returncode:
        print("\n".join(install_log.read_text(errors="replace").splitlines()[-50:]))
        raise RuntimeError("Installation failed; preserve the log.")

ninja = PY.parent / "ninja"
ninja_installed_now = not ninja.is_file()
if ninja_installed_now:
    subprocess.run([str(PY), "-m", "pip", "install", "--no-deps", "ninja"], check=True)

SERVER_ENV = os.environ.copy()
SERVER_ENV.pop("PYTHONPATH", None)
SERVER_ENV["PATH"] = str(PY.parent) + os.pathsep + SERVER_ENV.get("PATH", "")
SERVER_ENV["VIRTUAL_ENV"] = str(ENV)
SERVER_ENV["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
ninja_version = subprocess.check_output(["ninja", "--version"], env=SERVER_ENV, text=True).strip()

SETUP_REPORT, setup_text = child_json(r"""
import json, sys, torch, vllm
from importlib.metadata import version
available = torch.cuda.is_available()
print('UQROUTE_JSON=' + json.dumps({
 'python': sys.version, 'vllm': version('vllm'), 'torch': torch.__version__,
 'transformers': version('transformers'), 'cuda_available': available,
 'gpu_name': torch.cuda.get_device_name(0) if available else None,
 'gpu_memory_mib': round(torch.cuda.get_device_properties(0).total_memory / 1024**2) if available else None,
 'compute_capability': list(torch.cuda.get_device_capability(0)) if available else None
}))
""")
dep = subprocess.run([str(PY), "-m", "pip", "check"], capture_output=True, text=True)
if dep.returncode:
    print(dep.stdout + dep.stderr)
    raise RuntimeError("Dependency check failed.")
if not SETUP_REPORT["cuda_available"] or SETUP_REPORT["compute_capability"][0] < 8:
    raise RuntimeError("A native BF16-capable GPU is required.")
SETUP_REPORT.update(ninja_version=ninja_version, isolated_python=str(PY),
                    dependency_check_passed=True, ninja_binary_sha256=sha(ninja.read_bytes()),
                    ninja_installed_this_run=ninja_installed_now, held_out_access_enabled=False,
                    main_experiment_ready=False)
print(json.dumps(SETUP_REPORT, indent=2))
print("Environment ready; executable PATH includes:", PY.parent)


## 3. Verify the transferred bounded coverage plan

If the existing plan is absent, upload only `corrected_qwen15b_coverage_transfer_v1.zip` when prompted. No raw benchmark, audit inventory, reference sidecar or held-out data is uploaded.


In [ ]:
if not PREPARED.exists():
    archive_path = Path("/content/corrected_qwen15b_coverage_transfer_v1.zip")
    if not archive_path.is_file():
        from google.colab import files
        print("Choose corrected_qwen15b_coverage_transfer_v1.zip")
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise RuntimeError("Upload only the approved transfer ZIP.")
        name, raw_zip = next(iter(uploaded.items()))
        if not name.lower().endswith(".zip"):
            raise RuntimeError("Expected a ZIP file.")
        import io
        archive_source = io.BytesIO(raw_zip)
    else:
        archive_source = archive_path
    with zipfile.ZipFile(archive_source) as archive:
        if sorted(archive.namelist()) != ["manifest.json", "requests.json"] or archive.testzip() is not None:
            raise RuntimeError("Unexpected or damaged transfer archive.")
        payloads = {name: archive.read(name) for name in ("manifest.json", "requests.json")}
    if sha(payloads["manifest.json"]) != EXPECTED_MANIFEST:
        raise RuntimeError("Transfer manifest fingerprint differs.")
    PREPARED.mkdir(parents=True)
    for name, raw in payloads.items():
        with (PREPARED / name).open("xb") as stream:
            stream.write(raw)

if {p.name for p in PREPARED.iterdir()} != {"manifest.json", "requests.json"}:
    raise RuntimeError("Unexpected prepared directory contents; preserve it.")
if sha((PREPARED / "manifest.json").read_bytes()) != EXPECTED_MANIFEST:
    raise RuntimeError("Prepared manifest fingerprint differs.")
MANIFEST = json.loads((PREPARED / "manifest.json").read_bytes())
REQUESTS = json.loads((PREPARED / "requests.json").read_bytes())
if sha(canonical(REQUESTS)) != MANIFEST["requests_sha256"] or len(REQUESTS) != APPROVED_REQUEST_COUNT:
    raise RuntimeError("Prepared requests differ.")
if MANIFEST["selection"] != "coverage" or MANIFEST["held_out_access_enabled"] is not False:
    raise RuntimeError("Unexpected prepared scope.")
print("Transfer verification: PASSED; approved prepared coverage requests.")


## 4. Verify the pinned repository and frozen methods

The checkout is private to this Colab runtime. No fetch, reset, commit or push is applied to an existing checkout. A local Git exclusion keeps runtime artifacts out of staging. The benchmark repository and split data are not opened.


In [ ]:
if not REPO.exists():
    subprocess.run(["git", "clone", "--no-checkout", "--branch", "feat/task3-throughput", "--single-branch",
                    "https://github.com/RavindraSSK/uqroute-tc.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_REVISION], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
if head != CODE_REVISION:
    raise RuntimeError("Existing repository revision differs; preserve it.")
exclude = REPO / ".git/info/exclude"
rule = "/artifacts/local_development_audits/"
existing = exclude.read_text() if exclude.exists() else ""
if rule not in existing.splitlines():
    with exclude.open("a", encoding="utf-8") as stream:
        stream.write(("\n" if existing and not existing.endswith("\n") else "") + rule + "\n")
dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip()
if dirty:
    raise RuntimeError("Existing checkout has source changes; preserve it.")
META.mkdir(parents=True, exist_ok=True)
json_write_new(META / ("setup_" + uuid.uuid4().hex[:8] + ".json"), SETUP_REPORT)
run_workflow(["check-sources"])
validation = r"""
from pathlib import Path
import sys
repo, prepared = Path(sys.argv[1]), Path(sys.argv[2])
sys.path.insert(0, str(repo / 'tools/development_audits/corrected_generation_v1'))
from workflow import load_plan
manifest, requests = load_plan(repo, prepared, sys.argv[3])
if not 5 <= len(requests) <= 85 or manifest['selection'] != 'coverage':
    raise ValueError('Unexpected selection')
print('Plan verification: PASSED')
print('Model:', manifest['model_settings']['model_id'])
print('Dtype:', manifest['model_settings']['dtype'])
print('Benchmarks:', sorted({r['benchmark'] for r in requests}))
"""
subprocess.run([str(PY), "-c", validation, str(REPO), str(PREPARED), EXPECTED_MANIFEST],
               env=SERVER_ENV, check=True)
subprocess.run([str(PY), "-m", "unittest", "discover", "-s",
               str(REPO / "tools/development_audits/corrected_generation_v1"), "-p", "test_*.py"],
               env=SERVER_ENV, check=True)
print("Repository and frozen-method checks passed.")


## Context preflight before server startup
Render all approved prompts with the pinned tokenizer. Stop the entire run if any context limit is exceeded.


In [ ]:
# No generation request is sent by this preflight.
context_script = r"""

def context_report(prompt_token_counts, expected_rows, max_model_len=8192, completion_cap=1024):
    if not prompt_token_counts or len(prompt_token_counts) != expected_rows:
        raise ValueError('Prompt-token coverage differs')
    if type(max_model_len) is not int or type(completion_cap) is not int or max_model_len <= 0 or completion_cap <= 0:
        raise ValueError('Invalid context settings')
    if any(type(n) is not int or n < 1 for n in prompt_token_counts):
        raise ValueError('Invalid rendered token count')
    over = [i for i,n in enumerate(prompt_token_counts) if n + completion_cap > max_model_len]
    return {'rows': expected_rows, 'prompt_tokens_total': sum(prompt_token_counts),
            'prompt_tokens_max': max(prompt_token_counts), 'completion_cap': completion_cap,
            'max_model_len': max_model_len, 'over_limit_request_indices': over,
            'all_fit': not over, 'prompts_truncated': False, 'requests_dropped': 0,
            'limit': 'Pinned-tokenizer rendering preflight; server validation remains authoritative.'}

import json,sys
from pathlib import Path
from transformers import AutoTokenizer
prepared=Path(sys.argv[1]);manifest=json.loads((prepared/'manifest.json').read_text())
records=json.loads((prepared/'requests.json').read_text())
model=manifest['model_settings']
if model['model_id'] != 'Qwen/Qwen2.5-1.5B-Instruct':
    raise ValueError('Unexpected coverage model')
tokenizer=AutoTokenizer.from_pretrained(model['model_id'], revision=model['tokenizer_revision'])
counts=[]
for record in records:
    rendered=tokenizer.apply_chat_template(record['request']['messages'],tokenize=True,add_generation_prompt=True,return_dict=False)
    if not isinstance(rendered,list) or any(type(x) is not int for x in rendered):
        raise ValueError('Unexpected tokenizer return shape')
    counts.append(len(rendered))
report=context_report(counts,len(records),manifest['server_settings']['max_model_len'],manifest['generation_settings']['max_tokens'])
print('UQROUTE_CONTEXT='+json.dumps(report))
"""
result=subprocess.run([str(PY),'-c',context_script,str(PREPARED)],env=SERVER_ENV,capture_output=True,text=True)
if result.returncode:
    print(result.stdout);print(result.stderr)
    raise RuntimeError('Tokenizer preflight failed; no generation dispatched.')
marked=[line[len('UQROUTE_CONTEXT='):] for line in result.stdout.splitlines() if line.startswith('UQROUTE_CONTEXT=')]
if len(marked)!=1:raise RuntimeError('Missing context preflight report.')
CONTEXT_REPORT=json.loads(marked[0])
json_write_new(META/('context_preflight_'+uuid.uuid4().hex[:8]+'.json'),CONTEXT_REPORT)
print(json.dumps(CONTEXT_REPORT,indent=2))
if not CONTEXT_REPORT['all_fit']:
    raise RuntimeError('Context overflow: preserve plan/report. No truncation, exclusions or generation allowed.')


## 5. Start or reuse the pinned server

An existing durable intent is never sent again. If all coverage request intents already exist, the notebook skips server setup and goes directly to replay. Otherwise it starts or reuses the matching project server with the Ninja PATH fix. The readiness check may take several minutes while weights download and kernels compile. If it times out while the server remains alive, rerun this cell; it will not launch a duplicate matching server.


In [ ]:
ALL_INTENTS_PRESENT = all((RUN_DIR / "cases" / sha(r["case_key"].encode()) / "intent.json").is_file() for r in REQUESTS)
MODEL = MANIFEST["model_settings"]
SERVER_SETTINGS = MANIFEST["server_settings"]
COMMAND = [str(PY), "-m", "vllm.entrypoints.openai.api_server",
           "--model", MODEL["model_id"], "--revision", MODEL["model_revision"],
           "--tokenizer", MODEL["model_id"], "--tokenizer-revision", MODEL["tokenizer_revision"],
           "--served-model-name", MODEL["model_id"], "--dtype", MODEL["dtype"],
           "--max-model-len", str(SERVER_SETTINGS["max_model_len"]),
           "--gpu-memory-utilization", str(SERVER_SETTINGS["gpu_memory_utilization"]),
           "--host", "127.0.0.1", "--port", "8000"]


def server_models():
    try:
        with OPENER.open(ENDPOINT + "/models", timeout=3) as stream:
            return [x["id"] for x in json.load(stream).get("data", [])]
    except Exception:
        return None


def live_pid(pid):
    try:
        os.kill(int(pid), 0)
        return True
    except (OSError, ValueError):
        return False


def find_recorded_server():
    # Associate an existing project launch with a live API process recorded in its log.
    for record_path in sorted(ARTIFACTS.glob("qwen15b_server_*/launch.json"), key=lambda p:p.stat().st_mtime, reverse=True):
        record = json.loads(record_path.read_text())
        log = record_path.parent / "server.log"
        if record.get("command") != COMMAND or record.get("prepared_manifest_sha256") != EXPECTED_MANIFEST or not log.is_file():
            continue
        pids = re.findall(r"\(APIServer pid=(\d+)\)|Started server process \[(\d+)\]", log.read_text(errors="replace"))
        pids = [int(a or b) for a,b in pids]
        if any(live_pid(p) for p in pids):
            return record_path.parent
    return None

SERVER_OWNED_BY_NOTEBOOK = False
SERVER_READY = False
if ALL_INTENTS_PRESENT:
    print("All coverage request intents already exist. No server or generation restart is needed.")
else:
    known = find_recorded_server()
    if known is not None:
        SERVER_DIR = known
        print("Reusing recorded project server:", SERVER_DIR)
    else:
        with socket.socket() as sock:
            sock.settimeout(1)
            if sock.connect_ex(("127.0.0.1",8000)) == 0:
                raise RuntimeError("Port 8000 is occupied by a server without matching project evidence; it was left untouched.")
        SERVER_DIR = ARTIFACTS / ("qwen15b_server_" + uuid.uuid4().hex[:8])
        SERVER_DIR.mkdir()
        launch = {"started_utc": datetime.now(timezone.utc).isoformat(), "command": COMMAND,
                  "model_settings": MODEL, "server_settings": SERVER_SETTINGS,
                  "prepared_manifest_sha256": EXPECTED_MANIFEST, "model_requests_sent": 0,
                  "executable_path_prefix": str(PY.parent), "ninja_version": ninja_version,
                  "ninja_binary_sha256": sha(ninja.read_bytes()),
                  "environment_fix": "Isolated bin directory prepended to PATH; model and sampling settings unchanged"}
        json_write_new(SERVER_DIR / "launch.json", launch)
        with (SERVER_DIR / "server.log").open("xb") as stream:
            SERVER_PROCESS = subprocess.Popen(COMMAND, cwd=str(REPO), env=SERVER_ENV,
                                              stdout=stream, stderr=subprocess.STDOUT, start_new_session=True)
        SERVER_OWNED_BY_NOTEBOOK = True
        json_write_new(SERVER_DIR / "process.json", {"pid":SERVER_PROCESS.pid,"started_by_this_notebook":True})
        print("Server launched:", SERVER_PROCESS.pid)
    SERVER_LOG = SERVER_DIR / "server.log"
    deadline = time.monotonic() + 600
    last_notice = 0
    while time.monotonic() < deadline:
        names = server_models()
        if names is not None:
            if MODEL["model_id"] not in names:
                raise RuntimeError("Served model identifier differs.")
            SERVER_READY = True
            break
        if SERVER_OWNED_BY_NOTEBOOK and SERVER_PROCESS.poll() is not None:
            print("\n".join(SERVER_LOG.read_text(errors="replace").splitlines()[-140:]))
            raise RuntimeError("Server exited; preserve its log. No generation requests were sent by this notebook.")
        if time.monotonic()-last_notice > 20:
            print("Waiting for server; log:", SERVER_LOG, flush=True)
            last_notice = time.monotonic()
        time.sleep(2)
    if not SERVER_READY:
        print("\n".join(SERVER_LOG.read_text(errors="replace").splitlines()[-30:]))
        raise RuntimeError("Readiness timeout. If the process is still alive, rerun ONLY this cell.")
    print("SERVER READY:", MODEL["model_id"])
    print("Next, only slots without a prior intent can be dispatched.")


## 6. Record runtime observations

Requested temperature remains 0.001. The installed SamplingParams constructor is probed locally to record normalization (often 0.01); this is not a generation request. The server log snapshot and probe are retained for review. Revision/settings entries describe the recorded launch, not an independent hash verification of loaded GPU weights. Remaining Colab quota is explicitly unrecorded; this is only a bounded coverage feasibility run.


In [ ]:
RUNTIME_JSON = META / "runtime.json"
if ALL_INTENTS_PRESENT:
    if not RUNTIME_JSON.is_file():
        raise RuntimeError("Prior intents exist but the original runtime record is missing; preserve the run for inspection.")
    print("Using retained runtime evidence for CPU replay only.")
else:
    if not SERVER_READY:
        raise RuntimeError("Server readiness required before dispatch.")
    observed, probe_text = child_json(r"""
import json, torch
from vllm import SamplingParams
from importlib.metadata import version
params = SamplingParams(temperature=0.001, max_tokens=1024, logprobs=5)
print('UQROUTE_JSON=' + json.dumps({
 'vllm_version':version('vllm'), 'effective_temperature':params.temperature,
 'gpu_name':torch.cuda.get_device_name(0),
 'gpu_memory_mib':round(torch.cuda.get_device_properties(0).total_memory/1024**2)
}))
""")
    if RUNTIME_JSON.exists():
        # The existing binding must remain exact; never replace it on resume.
        retained = json.loads(RUNTIME_JSON.read_text())
        old_observations = json.loads((META / "runtime_observations.json").read_text())
        if Path(old_observations["server_launch_record"]).parent != SERVER_DIR:
            raise RuntimeError("Server instance changed since runtime binding; preserve the original run for review.")
        if any(retained[k] != observed[k] for k in observed):
            raise RuntimeError("Observed runtime differs from the original binding; preserve it for review.")
        print("Reusing unchanged runtime binding.")
    else:
        probe_path = META / "sampling_probe.log"
        with probe_path.open("x",encoding="utf-8") as stream:stream.write(probe_text)
        snapshot = META / "server_ready_snapshot.log"
        with snapshot.open("xb") as stream:stream.write(SERVER_LOG.read_bytes())
        command_record = META / "launch_argv.json"
        with command_record.open("xb") as stream:stream.write(canonical(COMMAND))
        runtime = {k:MODEL[k] for k in ("model_id","model_revision","tokenizer_revision","dtype","quantization")}
        runtime.update({k:SERVER_SETTINGS[k] for k in ("max_model_len","gpu_memory_utilization","chat_template")})
        runtime.update(observed)
        runtime.update(platform="Google Colab",
                       quota_note="Remaining GPU allocation quota not recorded; bounded coverage development feasibility only.",
                       launch_command_sha256=sha(command_record.read_bytes()),
                       server_log_sha256=sha(snapshot.read_bytes()))
        json_write_new(RUNTIME_JSON,runtime)
        json_write_new(META / "runtime_observations.json",{
            "sampling_temperature_source":"Installed SamplingParams constructor probe; inspect actual server warnings after requests",
            "sampling_probe_sha256":sha(probe_path.read_bytes()),
            "server_launch_record":str(SERVER_DIR / "launch.json"),
            "launch_record_sha256":sha((SERVER_DIR / "launch.json").read_bytes()),
            "runtime_identity_limit":"Recorded launch plus served model ID; loaded weights not independently authenticated"})
    print(json.dumps(json.loads(RUNTIME_JSON.read_text()),indent=2))


## 7. Run the approved coverage development slots

Running this cell explicitly dispatches the prepared requests. No new request is sent for any existing intent, including unknown or failed outcomes. Do not delete intents to obtain a retry. Official correctness is not scored here; output status, parser extraction and uncertainty availability stay separate.


In [ ]:
if ALL_INTENTS_PRESENT:
    print("All coverage intents already exist. Skipping network dispatch; proceeding to replay.")
else:
    run_workflow(["run", "--prepared", str(PREPARED),
                  "--expected-manifest-sha256", EXPECTED_MANIFEST,
                  "--runtime-json", str(RUNTIME_JSON), "--endpoint", ENDPOINT,
                  "--output", str(RUN_DIR), "--execute-development"])
print("Request phase finished. Previously recorded outcomes were not retried.")


## 8. Replay frozen evidence on CPU

Replay writes a new directory on each execution. Old response and validation files remain unchanged. Missing usage/unknown outcomes remain explicit; they are not zero cost. Token totals are server-reported counters, not verified monetary billing. No official score, calibrated confidence or routing result is produced.


In [ ]:
REPLAY_DIR = WORK / ("replay_" + uuid.uuid4().hex[:8])
run_workflow(["replay", "--prepared", str(PREPARED),
              "--expected-manifest-sha256", EXPECTED_MANIFEST,
              "--run-dir", str(RUN_DIR), "--output", str(REPLAY_DIR)])
SUMMARY = json.loads((REPLAY_DIR / "summary.json").read_text())
print("\nCOVERAGE SUMMARY")
for key in ("rows","attempted_slots","status_counts","whole_output_eligible",
            "meaningful_token_eligible","known_token_totals","missing_usage_attempts",
            "official_scoring","held_out_access_enabled","main_experiment_ready"):
    print(key + ":", SUMMARY[key])
if SUMMARY["rows"] != len(REQUESTS):
    raise RuntimeError("Unexpected replay population.")


## 9. Stop a server started by this notebook and download evidence

The archive contains only the prepared development plan, runtime/setup records, request outcomes, CPU replay and selected project server logs. It includes no model weights, raw benchmark, reference sidecar, held-out examples or Git history. It may contain private development prompts/model outputs: keep it as local project evidence, not a public Git artifact.

A newly launched server owned by this notebook is shut down after replay. A reused existing server is left running to avoid terminating another notebook's process; disconnect the runtime after downloading if it is no longer needed.


In [ ]:
if globals().get("SERVER_OWNED_BY_NOTEBOOK",False) and globals().get("SERVER_PROCESS") is not None:
    if SERVER_PROCESS.poll() is None:
        SERVER_PROCESS.terminate()
        try:SERVER_PROCESS.wait(timeout=30)
        except subprocess.TimeoutExpired:
            # Only the process group created by this notebook is targeted.
            os.killpg(SERVER_PROCESS.pid,signal.SIGTERM)
            print("Shutdown signal sent to this notebook's server process group.")
        print("This notebook's server shutdown requested.")
else:
    print("No newly owned server was stopped. A reused server may still be running.")

EVIDENCE_ZIP = Path("/content") / ("uqroute_qwen15b_corrected_coverage_" + uuid.uuid4().hex[:8] + ".zip")
with zipfile.ZipFile(EVIDENCE_ZIP,"x",zipfile.ZIP_DEFLATED) as archive:
    for name in ("manifest.json","requests.json"):
        archive.write(PREPARED/name,"prepared/"+name)
    for root,label in ((META,"metadata"),(RUN_DIR,"run"),(REPLAY_DIR,"replay")):
        for path in sorted(root.rglob("*")):
            if path.is_file() and not path.is_symlink():
                archive.write(path,label+"/"+path.relative_to(root).as_posix())
    observations = META/"runtime_observations.json"
    if observations.is_file():
        recorded=json.loads(observations.read_text())
        server_folder=Path(recorded["server_launch_record"]).parent
        if server_folder.resolve().is_relative_to(ARTIFACTS.resolve()):
            for name in ("launch.json","server.log","process.json"):
                path=server_folder/name
                if path.is_file() and not path.is_symlink():archive.write(path,"server/"+name)
            launch=json.loads((server_folder/"launch.json").read_text())
            fix_path=launch.get("build_tool_fix_record")
            if fix_path:
                path=Path(fix_path)
                if path.resolve().is_relative_to(ARTIFACTS.resolve()) and path.is_file():
                    archive.write(path,"server/build_tool_fix.json")
with zipfile.ZipFile(EVIDENCE_ZIP) as archive:
    if archive.testzip() is not None:raise RuntimeError("Evidence ZIP integrity check failed.")
print("Evidence ZIP:",EVIDENCE_ZIP)
print("Evidence SHA256:",sha(EVIDENCE_ZIP.read_bytes()))
print("Official scoring: not performed. Main experiment ready: false.")
from google.colab import files
files.download(str(EVIDENCE_ZIP))


## Completed checkpoint

Save the downloaded evidence ZIP and the aggregate feasibility summary. Do not repeat generation to improve a result: prior failed/unknown slots are retained. Next review response status, byte evidence and format eligibility, then perform a separate CPU official-scoring step against unchanged development references. Do not claim accuracy from this notebook's parser/availability counts.

Held-out evaluation remains disabled. After the download completes, disconnect/delete the GPU runtime if it is no longer needed.
